In [38]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [39]:
# Load the trained model, scaler pickle,onehot
model = load_model('model.keras')

#load the encoder and scaler
with open('onehot_encoder_geo.pkl', 'rb') as file:
    label_encoder_geo=pickle.load(file)

with open('label_encoder.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

/Users/prabhatkumar/Documents/ANN Classification /venv/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 8 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [40]:
# example input data 
input_data = {
    'creditscore': 600,
    'Geography': 'France',
    'gender': 'Male',
    'age': 40,
    'tenure': 3,
    'balance': 60000,
    'numofproducts': 2,
    'hascrcard': 1,         
    'isactivemember': 1,    
    'estimatedsalary': 50000
}

In [41]:
# one hot encode geography 
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

 # Assuming 'geography' is the variable containing the input value

/Users/prabhatkumar/Documents/ANN Classification /venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [42]:
input_df=pd.DataFrame([input_data])
input_df


,creditscore,Geography,gender,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [45]:
#Encoded categorical variables
input_df['gender'] = label_encoder_gender.transform(input_df['gender']) 
input_df


,creditscore,Geography,gender,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary
0,600,France,1,40,3,60000,2,1,1,50000


In [46]:
# concatination one hot encoded
input_df = pd.concat([input_df.drop("Geography", axis=1), geo_encoded_df], axis=1)
input_df

,creditscore,gender,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [52]:
# Scaling the input data
print("Scaler expects:", scaler.feature_names_in_)
print("Input has:", input_df.columns)


Scaler expects: ['CreditScore' 'Gender' 'Age' 'Tenure' 'Balance' 'NumOfProducts'
 'HasCrCard' 'IsActiveMember' 'EstimatedSalary' 'Geography_France'
 'Geography_Germany' 'Geography_Spain']
Input has: Index(['creditscore', 'gender', 'age', 'tenure', 'balance', 'numofproducts',
       'hascrcard', 'isactivemember', 'estimatedsalary', 'Geography_France',
       'Geography_Germany', 'Geography_Spain'],
      dtype='object')


In [53]:
# Add missing columns if any
for col in scaler.feature_names_in_:
    if col not in input_df.columns:
        input_df[col] = 0

# Remove extra columns
input_df = input_df[scaler.feature_names_in_]

# Now scale
input_scaled = scaler.transform(input_df)

In [54]:
# Scaling the input data
scaled_input = scaler.transform(input_df)
input_scaled

array([[-6.76262379, -1.09499335, -3.69810386, -1.73646664, -1.21847056,
        -2.6418115 , -1.54035103, -1.02583358, -1.74616572,  1.00150113,
        -0.57946723, -0.57638802]])

In [57]:
# prediction churn 
prediction = model.predict(input_scaled)
prediction




1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step


array([[0.03570598]], dtype=float32)

In [60]:
prediction_proba = prediction[0][0]




In [61]:
prediction_proba

0.035705984

In [62]:
if prediction_proba > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is not likely to churn.")   
    

The customer is not likely to churn.
